# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access basic information (dataset.metadata is an object)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Date published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and label (name)
print("Available record sets:")
for rs in dataset.metadata.record_sets:
    print(f"  - @id: {rs['@id']} | name: {rs.get('name', rs['@id'])}")

# Explore the fields within each record set
for rs in dataset.metadata.record_sets:
    print(f"\nRecord set @id: {rs['@id']} ({rs.get('name', rs['@id'])})")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', field['@id'])}, dataType: {field.get('dataType', 'Unknown')}")
    else:
        print("  No fields found in this record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract data from all available record sets
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading record set {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}")

if dataframes:
    main_record_set = list(dataframes.keys())[0]  # Choose the first available
    print(f"\nFields (@id) in main record set ({main_record_set}):")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Update these @id references as needed based on your record set/columns
df = dataframes[main_record_set]

# Try to find a numeric field
numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_candidates:
    print("No numeric columns detected.")
    # Maybe they are loaded as string, check columns for numeric-like names
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col])
            numeric_candidates.append(col)
        except Exception:
            pass

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field for analysis: {numeric_field}")
    # Attempt cleaning
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    # Filter out nulls
    filtered_df = df[df[numeric_field].notnull()]
    mean_value = filtered_df[numeric_field].mean()
    std_value = filtered_df[numeric_field].std()
    threshold = mean_value if mean_value > 0 else 1  # Simple threshold
    high_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(high_df)} records")
    display(high_df.head())
    # Normalize
    high_df[f"{numeric_field}_normalized"] = (high_df[numeric_field] - mean_value) / std_value
    print(f"Normalized {numeric_field} for filtered records:")
    display(high_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try grouping by a categorical column
    # Find a possible category field (not numeric)
    group_field_candidates = [col for col in df.columns if col != numeric_field and not np.issubdtype(df[col].dtype, np.number)]
    group_field = None
    if group_field_candidates:
        group_field = group_field_candidates[0]
    if group_field and group_field in high_df.columns:
        grouped_df = high_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    # Plot histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we identified a group field, show boxplot
    if group_field:
        plt.figure(figsize=(12, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library, explored the available record sets, fields (with their `@id`s), and extracted sample records into pandas DataFrames.

- We identified numeric columns and demonstrated basic filtering and normalization.
- Exploratory visualizations reveal the distribution of a quantitative field and its relation to a categorical field, where available.

You can extend this notebook for deeper analysis—such as comparing groups, applying additional transformations, or training models—using the field and record set `@id`s for robust and reproducible data access.

**Remember:** Always reference record sets, fields, and columns using their `@id` attributes, as shown throughout this notebook.